# NS_PIKANS_geo_BASELINE - Pipeline experimental limpio

## 0. Notebook contract

**Objetivo.** Comparar `FCN`, `PIKAN-NORM` y `cPIKAN-NORM` en Navier-Stokes 2D estacionario alrededor de un cilindro.

**Regla experimental.** Mantener la misma geometria, mismos puntos de frontera, mismos puntos de colocacion, misma loss y mismos optimizadores para todos los experimentos.

**Unico cambio entre corridas.** La arquitectura del aproximador neuronal.

Este notebook esta organizado para ejecutarse de arriba hacia abajo, sin depender de celdas antiguas, dropdowns ni variables globales implicitas en las funciones criticas.

## 1. Imports minimos y seed

In [ ]:
import os
import json
import time
import random
from dataclasses import asdict, dataclass
from collections import OrderedDict

import numpy as np
import torch
from torch import nn
from torch.quasirandom import SobolEngine
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Configs formales

In [ ]:
@dataclass(frozen=True)
class DomainConfig:
    width: float = 2.2
    height: float = 0.41
    circle_center_x: float = 0.2
    circle_center_y: float = 0.2
    circle_radius: float = 0.05
    nu: float = 0.001
    U: float = 0.3


@dataclass(frozen=True)
class DataConfig:
    n_train_bc: int = 100
    n_circle_bc: int = 40
    n_train_coll: int = 1500
    n_circle_coll: int = 50
    seed: int = 42

    # Puntos extra dirigidos
    n_near_cylinder: int = 200
    n_wake: int = 200

    # Pesos BC por región
    w_bc_circle: float = 5.0
    w_bc_walls: float = 1.0
    w_bc_inlet: float = 3.0


@dataclass(frozen=True)
class TrainConfig:
    epochs_adam: int = 2000
    epochs_lbfgs: int = 1000
    lr_adam: float = 1e-3
    lr_lbfgs: float = 0.01
    checkpoint_every: int = 200
    resume: bool = True
    auto_weight_adam: bool = False
    auto_weight_lbfgs: bool = False
    loss_weight_update_frequency: int = 500
    lbfgs_max_iter: int = 20
    lbfgs_history_size: int = 100
    allow_incompatible_resume: bool = False


domain_cfg = DomainConfig()
data_cfg = DataConfig(seed=SEED)
train_cfg = TrainConfig()

## 3. Paths

Mantiene Drive cuando el notebook corre en Colab. Cada experimento guarda artefactos en subcarpetas propias para evitar checkpoints incompatibles entre arquitecturas.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    root = '/content/drive/MyDrive/GradProject/NS'
except Exception:
    root = os.path.abspath('./GradProject')

model_root = os.path.join(root, 'NS_models')
results_root = os.path.join(root, 'NS_results')
os.makedirs(model_root, exist_ok=True)
os.makedirs(results_root, exist_ok=True)

print(f'root        : {root}')
print(f'model_root  : {model_root}')
print(f'results_root: {results_root}')

## 4. Geometry and data generation

La logica fisica se mantiene: canal 2D, cilindro, paredes superior/inferior no-slip, entrada parabolica y puntos de colocacion fuera del cilindro. Las funciones reciben `domain_cfg` y `data_cfg` en vez de leer globales.

In [ ]:
def inside_circle_condition(x_orig, y_orig, r, x, y):
    return (x - x_orig) ** 2 + (y - y_orig) ** 2 < r ** 2


def circle_condition(x_orig, y_orig, r, x, y):
    return (x - x_orig) ** 2 + (y - y_orig) ** 2 == r ** 2


def _lhs_unit(n, dim, seed):
    rng = np.random.default_rng(seed)
    result = np.empty((n, dim), dtype=np.float64)
    cut = np.linspace(0.0, 1.0, n + 1)
    for j in range(dim):
        points = cut[:-1] + rng.random(n) * (cut[1:] - cut[:-1])
        rng.shuffle(points)
        result[:, j] = points
    return result


def set_circle_data(domain_cfg, n_circle):
    theta = np.linspace(0, 2 * np.pi, n_circle)
    x_circle = domain_cfg.circle_center_x + domain_cfg.circle_radius * np.cos(theta)
    y_circle = domain_cfg.circle_center_y + domain_cfg.circle_radius * np.sin(theta)
    x_circle = np.hstack((x_circle[:, None], y_circle[:, None]))
    u_circle = np.zeros(x_circle.shape)
    return x_circle, u_circle


def inflow(y, domain_cfg):
    return (4 * domain_cfg.U * y * (domain_cfg.height - y)) / (domain_cfg.height ** 2)


def set_lower_and_upper_walls_boundary(x, y):
    x_up = np.hstack([x[0, :][:, None], y[0, :][:, None]])
    u_up = np.zeros(x_up.shape)

    x_low = np.hstack([x[-1, :][:, None], y[-1, :][:, None]])
    u_low = np.zeros(x_low.shape)

    return x_up, u_up, x_low, u_low


def set_left_wall_boundary(x, y, domain_cfg):
    x_left = np.hstack([x[:, 0][:, None], y[:, 0][:, None]])
    u_left_y = inflow(x_left[:, 1], domain_cfg)
    u_left = np.hstack([u_left_y[:, None], np.zeros_like(u_left_y)[:, None]])
    return x_left, u_left


def get_boundary_conditions(domain_cfg, data_cfg, normalize=False):
    # Fix: sin meshgrid — LHS directo sobre las paredes
    sample = _lhs_unit(data_cfg.n_train_bc, 2, data_cfg.seed)
    x_wall = sample[:, 0] * domain_cfg.width
    y_wall = sample[:, 1] * domain_cfg.height

    x_circle, u_circle = set_circle_data(domain_cfg, data_cfg.n_circle_bc)

    # Paredes superior e inferior: y fijo, x distribuido
    x_up  = np.hstack([x_wall[:, None],
                        np.full((len(x_wall), 1), domain_cfg.height)])
    u_up  = np.zeros_like(x_up)
    x_low = np.hstack([x_wall[:, None], np.zeros((len(x_wall), 1))])
    u_low = np.zeros_like(x_low)

    # Pared izquierda: x=0, y distribuido
    x_left     = np.hstack([np.zeros((len(y_wall), 1)), y_wall[:, None]])
    u_left_y   = inflow(y_wall, domain_cfg)
    u_left     = np.hstack([u_left_y[:, None], np.zeros_like(u_left_y)[:, None]])

    points = np.vstack([x_circle, x_up, x_low, x_left])
    values = np.vstack([u_circle, u_up, u_low, u_left])
    p_column = np.zeros(points.shape[0])
    values = np.hstack([values, p_column[:, None]])

    mask = ~inside_circle_condition(
        domain_cfg.circle_center_x, domain_cfg.circle_center_y,
        domain_cfg.circle_radius, points[:, 0], points[:, 1],
    )
    points = points[mask]
    values = values[mask]

    if normalize:
        points[:, 0] = 2.0 * points[:, 0] / domain_cfg.width  - 1.0
        points[:, 1] = 2.0 * points[:, 1] / domain_cfg.height - 1.0

    x_bc = torch.tensor(points, dtype=torch.float32, requires_grad=True, device=device)
    y_bc = torch.tensor(values, dtype=torch.float32, device=device)
    return x_bc, y_bc


def get_collocation_points(domain_cfg, data_cfg, normalize=False):
    sobol  = SobolEngine(dimension=2, scramble=True, seed=data_cfg.seed)
    sample = sobol.draw(data_cfg.n_train_coll).cpu().numpy()
    x = sample[:, 0] * domain_cfg.width
    y = sample[:, 1] * domain_cfg.height
    points = np.hstack((x[:, None], y[:, None]))

    mask = ~inside_circle_condition(
        domain_cfg.circle_center_x, domain_cfg.circle_center_y,
        domain_cfg.circle_radius, points[:, 0], points[:, 1],
    )
    points = points[mask]

    # Puntos extra cerca del cilindro — radio expandido 3x
    sobol2 = SobolEngine(dimension=2, scramble=True, seed=data_cfg.seed + 1)
    n_near = data_cfg.n_near_cylinder
    s2     = sobol2.draw(n_near * 4).cpu().numpy()
    cx, cy, r = (domain_cfg.circle_center_x,
                 domain_cfg.circle_center_y,
                 domain_cfg.circle_radius)
    x_near = cx + (s2[:, 0] * 2 - 1) * r * 3
    y_near = cy + (s2[:, 1] * 2 - 1) * r * 3
    near   = np.hstack((x_near[:, None], y_near[:, None]))
    mask_near = (
        ~inside_circle_condition(cx, cy, r, near[:, 0], near[:, 1]) &
        (near[:, 0] >= 0) & (near[:, 0] <= domain_cfg.width) &
        (near[:, 1] >= 0) & (near[:, 1] <= domain_cfg.height)
    )
    near = near[mask_near][:n_near]

    # Puntos extra en la estela — caja detrás del cilindro
    sobol3  = SobolEngine(dimension=2, scramble=True, seed=data_cfg.seed + 2)
    n_wake = data_cfg.n_wake
    s3      = sobol3.draw(n_wake * 2).cpu().numpy()
    x_wake  = cx + r + s3[:, 0] * r * 8       # hasta 8r detrás
    y_wake  = cy + (s3[:, 1] * 2 - 1) * r * 3
    wake    = np.hstack((x_wake[:, None], y_wake[:, None]))
    mask_wake = (
        (wake[:, 0] >= 0) & (wake[:, 0] <= domain_cfg.width) &
        (wake[:, 1] >= 0) & (wake[:, 1] <= domain_cfg.height)
    )
    wake = wake[mask_wake][:n_wake]

    # Puntos del cilindro (superficie)
    x_circle, _ = set_circle_data(domain_cfg, data_cfg.n_circle_coll)

    points = np.vstack((points, near, wake, x_circle))

    if normalize:
        points[:, 0] = 2.0 * points[:, 0] / domain_cfg.width  - 1.0
        points[:, 1] = 2.0 * points[:, 1] / domain_cfg.height - 1.0

    x_coll = torch.tensor(points[:, 0:1], dtype=torch.float32,
                           requires_grad=True, device=device)
    y_coll = torch.tensor(points[:, 1:2], dtype=torch.float32,
                           requires_grad=True, device=device)
    return x_coll, y_coll

def get_test_grid(domain_cfg, n_test=300):
    sobol = SobolEngine(dimension=2, scramble=False, seed=0)
    sample = sobol.draw(n_test).cpu().numpy()
    x = sample[:, 0] * domain_cfg.width
    y = sample[:, 1] * domain_cfg.height
    mask = ~inside_circle_condition(
        domain_cfg.circle_center_x,
        domain_cfg.circle_center_y,
        domain_cfg.circle_radius,
        x,
        y,
    )
    points = np.hstack((x[mask, None], y[mask, None]))
    return torch.tensor(points, dtype=torch.float32, device=device), points

In [ ]:
# FCN no normaliza — datos en coordenadas físicas
x_bc_phys,  y_bc_phys  = get_boundary_conditions(domain_cfg, data_cfg, normalize=False)
x_coll_phys, y_coll_phys = get_collocation_points(domain_cfg, data_cfg, normalize=False)

# KANs normalizan — datos en [-1,1]²
x_bc_norm,  y_bc_norm  = get_boundary_conditions(domain_cfg, data_cfg, normalize=True)
x_coll_norm, y_coll_norm = get_collocation_points(domain_cfg, data_cfg, normalize=True)

tensors_phys = {
    'x_bc': x_bc_phys, 'y_bc': y_bc_phys,
    'x_coll': x_coll_phys, 'y_coll': y_coll_phys,
}
tensors_norm = {
    'x_bc': x_bc_norm, 'y_bc': y_bc_norm,
    'x_coll': x_coll_norm, 'y_coll': y_coll_norm,
}

print(f'BC points (phys) : {x_bc_phys.shape[0]}')
print(f'Coll points      : {x_coll_phys.shape[0]}')

In [ ]:
def plot_initial_conditions(x_bc, y_bc, x_coll, y_coll, domain_cfg):
    x_bc_np   = x_bc.detach().cpu().numpy()
    y_bc_np   = y_bc.detach().cpu().numpy()
    x_coll_np = x_coll.detach().cpu().numpy()
    y_coll_np = y_coll.detach().cpu().numpy()

    theta = np.linspace(0, 2*np.pi, 100)
    xc = domain_cfg.circle_center_x + domain_cfg.circle_radius * np.cos(theta)
    yc = domain_cfg.circle_center_y + domain_cfg.circle_radius * np.sin(theta)

    def style_ax(ax, title):
        ax.set_facecolor('#f8fafc')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#cbd5e1')
        ax.spines['bottom'].set_color('#cbd5e1')
        ax.tick_params(colors='#475569', labelsize=8)
        ax.set_xlabel('X', color='#475569', fontsize=9)
        ax.set_ylabel('Y', color='#475569', fontsize=9)
        ax.set_title(title, color='#0f172a', fontsize=10,
                     fontweight='bold', pad=6)
        ax.fill(xc, yc, color='#1e293b', alpha=0.85, zorder=5)
        ax.plot(xc, yc, color='#0f172a', lw=1, zorder=6)
        ax.set_xlim(-0.02, domain_cfg.width  + 0.02)
        ax.set_ylim(-0.01, domain_cfg.height + 0.01)

    fig = plt.figure(figsize=(20, 8))
    fig.patch.set_facecolor('white')
    gs  = fig.add_gridspec(2, 2, hspace=0.5, wspace=0.25)

    # ── Panel 1: Puntos de colocación ────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    style_ax(ax1, f'Puntos de colocación — Sobol  (n={x_coll_np.shape[0]:,})')
    ax1.scatter(x_coll_np, y_coll_np, s=4, c='#2563eb',
                alpha=0.6, linewidths=0)

    # ── Panel 2: BC — velocidad u ────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    style_ax(ax2, f'BC — Velocidad horizontal u  (n={x_bc_np.shape[0]:,})')
    sc2 = ax2.scatter(x_bc_np[:, 0], x_bc_np[:, 1],
                      c=y_bc_np[:, 0], cmap='turbo',
                      s=18, linewidths=0)
    cb2 = plt.colorbar(sc2, ax=ax2, fraction=0.03, pad=0.02)
    cb2.ax.tick_params(labelsize=8, colors='#475569')
    cb2.set_label(r'$u^{(x)}$', color='#0f172a', fontsize=9)

    # ── Panel 3: BC — velocidad v ────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    style_ax(ax3, 'BC — Velocidad vertical v')
    sc3 = ax3.scatter(x_bc_np[:, 0], x_bc_np[:, 1],
                      c=y_bc_np[:, 1], cmap='coolwarm',
                      s=18, linewidths=0)
    cb3 = plt.colorbar(sc3, ax=ax3, fraction=0.03, pad=0.02)
    cb3.ax.tick_params(labelsize=8, colors='#475569')
    cb3.set_label(r'$u^{(y)}$', color='#0f172a', fontsize=9)

    # ── Panel 4: BC — presión p ──────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 1])
    style_ax(ax4, 'BC — Presión p (inicializada en cero)')
    sc4 = ax4.scatter(x_bc_np[:, 0], x_bc_np[:, 1],
                      c=y_bc_np[:, 2], cmap='plasma',
                      s=18, linewidths=0)
    cb4 = plt.colorbar(sc4, ax=ax4, fraction=0.03, pad=0.02)
    cb4.ax.tick_params(labelsize=8, colors='#475569')
    cb4.set_label(r'$p$', color='#0f172a', fontsize=9)

    fig.suptitle('Condiciones iniciales — NS cilindro 2D',
                 color='#0f172a', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(results_root, 'initial_conditions.pdf'),
                format='pdf', bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  BC points   : {x_bc_np.shape[0]:,}")
    print(f"  Coll points : {x_coll_np.shape[0]:,}")

plot_initial_conditions(x_bc_phys, y_bc_phys, x_coll_phys, y_coll_phys, domain_cfg)

## 5. Architectures

Se mantienen `FCN`, `BSplineKANLayer`, `PIKAN`, `ChebyshevKANLayer` y `cPIKAN`. La normalizacion de entrada queda integrada como `InputNormalizer` con buffers registrados.

In [ ]:
class InputNormalizer(nn.Module):
    """Conservada como clase pero ya no se usa en PIKAN/cPIKAN.
    Los datos llegan normalizados desde get_collocation_points."""
    def __init__(self, width, height):
        super().__init__()
        self.register_buffer('x_min', torch.tensor([0.0, 0.0]))
        self.register_buffer('x_max', torch.tensor([width, height]))

    def forward(self, x):
        return 2.0 * (x - self.x_min) / (self.x_max - self.x_min) - 1.0


class FCN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        activation = nn.Tanh
        layers = OrderedDict([
            ('input', nn.Linear(input_size, hidden_size)),
            ('act0', activation()),
        ])
        for i in range(num_layers):
            layers[f'hidden_{i}'] = nn.Linear(hidden_size, hidden_size)
            layers[f'act_{i}']    = activation()
        layers['output'] = nn.Linear(hidden_size, output_size)
        self.net = nn.Sequential(layers)

    def forward(self, x):
        return self.net(x)


class BSplineKANLayer(nn.Module):
    def __init__(self, in_features, out_features, degree=3,
                 n_knots=5, grid_range=(-1, 1)):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.degree       = degree
        self.n_knots      = n_knots

        knots_interior = torch.linspace(grid_range[0], grid_range[1], n_knots + 1)
        knots = torch.cat([
            knots_interior[0].repeat(degree),
            knots_interior,
            knots_interior[-1].repeat(degree),
        ])
        self.register_buffer('knots', knots)
        self.n_basis = len(knots) - degree - 1

        # Cambio 1: escala de inicialización más pequeña
        self.coeff    = nn.Parameter(
            torch.randn(in_features, out_features, self.n_basis) * 0.01
        )
        self.w_base   = nn.Parameter(torch.ones(in_features, out_features) * 0.1)
        self.w_spline = nn.Parameter(torch.ones(in_features, out_features) * 0.1)

    def b_spline_basis(self, x):
        k   = self.knots
        x_e = x.unsqueeze(-1)
        B   = ((x_e >= k[:-1].view(1, 1, -1)) &
               (x_e <  k[1: ].view(1, 1, -1))).float()
        # Incluir el extremo derecho en el último intervalo
        B = torch.where(
            (x_e == k[-1]) & (
                torch.arange(B.shape[-1], device=x.device).view(1, 1, -1)
                == B.shape[-1] - 1
            ),
            torch.ones_like(B), B,
        )
        for p in range(1, self.degree + 1):
            n_curr  = B.shape[-1] - 1
            B_left  = B[..., :n_curr]
            B_right = B[..., 1:n_curr + 1]
            t_i   = k[:n_curr        ].view(1, 1, -1)
            t_ip  = k[p:p + n_curr   ].view(1, 1, -1)
            t_i1  = k[1:n_curr + 1   ].view(1, 1, -1)
            t_ip1 = k[p+1:p+1+n_curr ].view(1, 1, -1)
            denom1 = (t_ip  - t_i ).clamp(min=1e-8)
            denom2 = (t_ip1 - t_i1).clamp(min=1e-8)
            B = ((x_e - t_i) / denom1) * B_left + \
                ((t_ip1 - x_e) / denom2) * B_right
        return B

    def forward(self, x):
        # Cambio 2: tanh antes de evaluar splines — garantiza input en [-1,1]
        x_safe = torch.tanh(x)
        base       = torch.nn.functional.silu(x)
        B          = self.b_spline_basis(x_safe)
        spline_out = torch.einsum('nib,iob->nio', B, self.coeff)
        return (self.w_base * base.unsqueeze(2) +
                self.w_spline * spline_out).sum(dim=1)


class PIKAN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size,
                 degree=3, n_knots=5, **kwargs):
        super().__init__()
        # Cambio 7: sin normalizer — datos ya llegan normalizados
        dims = [input_size] + [hidden_size] * num_layers + [output_size]
        self.layers = nn.ModuleList([
            BSplineKANLayer(dims[i], dims[i+1], degree=degree, n_knots=n_knots)
            for i in range(len(dims) - 1)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class ChebyshevKANLayer(nn.Module):
    def __init__(self, in_features, out_features, degree=3):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.degree       = degree
        # Cambio 1: escala pequeña
        self.coeff  = nn.Parameter(
            torch.randn(in_features, out_features, degree + 1) * 0.01
        )
        self.w_base = nn.Parameter(
            torch.ones(in_features, out_features) * 0.1
        )

    def chebyshev_basis(self, x):
        # tanh garantiza dominio [-1,1] para Chebyshev
        x_norm = torch.tanh(x)
        polys  = [torch.ones_like(x_norm), x_norm]
        for _ in range(2, self.degree + 1):
            polys.append(2 * x_norm * polys[-1] - polys[-2])
        return torch.stack(polys[:self.degree + 1], dim=-1)

    def forward(self, x):
        base     = torch.nn.functional.silu(x)
        T        = self.chebyshev_basis(x)
        cheb_out = torch.einsum('nid,iod->nio', T, self.coeff)
        return (self.w_base * base.unsqueeze(2) + cheb_out).sum(dim=1)


class cPIKAN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size,
                 degree=3, **kwargs):
        super().__init__()
        # Cambio 7: sin normalizer
        dims = [input_size] + [hidden_size] * num_layers + [output_size]
        self.layers = nn.ModuleList([
            ChebyshevKANLayer(dims[i], dims[i+1], degree=degree)
            for i in range(len(dims) - 1)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

## 6. Model registry

`MODEL_CONFIGS` es la unica fuente de verdad. No se usan nombres antiguos ni registros paralelos.

In [ ]:
def build_experiment_configs(domain_cfg):
    return {
        # ── FCN baseline ─────────────────────────────────────────
        'FCN': {
            'factory'   : lambda: FCN(2, 50, 4, 3),
            'family'    : 'FCN',
            'hidden'    : 50, 'layers': 4, 'degree': None,
            'normalized': False,
            'tensors'   : 'phys',
        },
        # ── B-spline PIKAN ───────────────────────────────────────
        'PIKAN-d5': {
            'factory'   : lambda: PIKAN(2, 32, 3, 3, degree=5),
            'family'    : 'PIKAN',
            'hidden'    : 32, 'layers': 3, 'degree': 5,
            'normalized': True, 'tensors': 'norm',
        },
        # ── cPIKAN (Chebyshev) ───────────────────────────────────
        'cPIKAN-d5': {
            'factory'   : lambda: cPIKAN(2, 32, 3, 3, degree=5),
            'family'    : 'cPIKAN',
            'hidden'    : 32, 'layers': 3, 'degree': 5,
            'normalized': True, 'tensors': 'norm',
        },
    }

MODEL_CONFIGS = build_experiment_configs(domain_cfg)
print(list(MODEL_CONFIGS.keys()))

## 7. Losses

Las funciones reciben `model` explicitamente. El residual Navier-Stokes estacionario se mantiene con las mismas componentes: momento en `x`, momento en `y` y continuidad.

In [ ]:
def derivative_scales(domain_cfg, normalized=False):
    """
    Si x,y están normalizados a [-1,1], entonces:
        x_phys = width/2 * (x_norm + 1)
        y_phys = height/2 * (y_norm + 1)

    Por chain rule:
        d/dx_phys = (2/width)  d/dx_norm
        d/dy_phys = (2/height) d/dy_norm
    """
    if not normalized:
        sx = 1.0
        sy = 1.0
    else:
        sx = 2.0 / domain_cfg.width
        sy = 2.0 / domain_cfg.height

    return sx, sy


def physics_loss(model, x_coll, y_coll, domain_cfg, normalized=False):
    data = torch.hstack((x_coll, y_coll))
    output = model(data)

    u = output[:, 0:1]
    v = output[:, 1:2]
    p = output[:, 2:3]

    sx, sy = derivative_scales(domain_cfg, normalized=normalized)

    # Derivadas respecto a las coordenadas que entran al modelo
    ux_raw = torch.autograd.grad(
        u, x_coll,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    uy_raw = torch.autograd.grad(
        u, y_coll,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    vx_raw = torch.autograd.grad(
        v, x_coll,
        grad_outputs=torch.ones_like(v),
        create_graph=True
    )[0]
    vy_raw = torch.autograd.grad(
        v, y_coll,
        grad_outputs=torch.ones_like(v),
        create_graph=True
    )[0]

    px_raw = torch.autograd.grad(
        p, x_coll,
        grad_outputs=torch.ones_like(p),
        create_graph=True
    )[0]
    py_raw = torch.autograd.grad(
        p, y_coll,
        grad_outputs=torch.ones_like(p),
        create_graph=True
    )[0]

    # Convertir a derivadas físicas
    ux = sx * ux_raw
    uy = sy * uy_raw
    vx = sx * vx_raw
    vy = sy * vy_raw
    px = sx * px_raw
    py = sy * py_raw

    # Segundas derivadas: también necesitan chain rule
    uxx_raw = torch.autograd.grad(
        ux_raw, x_coll,
        grad_outputs=torch.ones_like(ux_raw),
        create_graph=True
    )[0]
    uyy_raw = torch.autograd.grad(
        uy_raw, y_coll,
        grad_outputs=torch.ones_like(uy_raw),
        create_graph=True
    )[0]

    vxx_raw = torch.autograd.grad(
        vx_raw, x_coll,
        grad_outputs=torch.ones_like(vx_raw),
        create_graph=True
    )[0]
    vyy_raw = torch.autograd.grad(
        vy_raw, y_coll,
        grad_outputs=torch.ones_like(vy_raw),
        create_graph=True
    )[0]

    uxx = (sx ** 2) * uxx_raw
    uyy = (sy ** 2) * uyy_raw
    vxx = (sx ** 2) * vxx_raw
    vyy = (sy ** 2) * vyy_raw

    ns_x = -domain_cfg.nu * (uxx + uyy) + u * ux + v * uy + px
    ns_y = -domain_cfg.nu * (vxx + vyy) + u * vx + v * vy + py
    continuity = ux + vy

    return (
        torch.mean(ns_x ** 2)
        + torch.mean(ns_y ** 2)
        + torch.mean(continuity ** 2)
    )


def denormalize_points(x, domain_cfg, normalized=False):
    """
    x: tensor (N, 2)
    Retorna coordenadas físicas para construir máscaras geométricas.
    """
    if not normalized:
        return x

    x_phys = x.clone()
    x_phys[:, 0] = (x[:, 0] + 1.0) * domain_cfg.width / 2.0
    x_phys[:, 1] = (x[:, 1] + 1.0) * domain_cfg.height / 2.0
    return x_phys


def build_bc_regions(x_bc, domain_cfg, data_cfg, normalized=False):
    """
    Construye máscaras por región usando coordenadas físicas.
    Funciona tanto si x_bc está físico como si está normalizado.
    """
    x_phys = denormalize_points(x_bc.detach(), domain_cfg, normalized=normalized)

    x = x_phys[:, 0]
    y = x_phys[:, 1]

    tol = 0.02
    cx = domain_cfg.circle_center_x
    cy = domain_cfg.circle_center_y
    r = domain_cfg.circle_radius

    mask_inlet = x < tol
    mask_top = y > domain_cfg.height - tol
    mask_bottom = y < tol

    # La superficie del cilindro está en los BC.
    # Usamos banda tolerante para capturar esos puntos.
    dist2 = (x - cx) ** 2 + (y - cy) ** 2
    mask_circle = dist2 <= (r * 1.25) ** 2

    return {
        "inlet": (mask_inlet, data_cfg.w_bc_inlet),
        "top": (mask_top, data_cfg.w_bc_walls),
        "bottom": (mask_bottom, data_cfg.w_bc_walls),
        "circle": (mask_circle, data_cfg.w_bc_circle),
    }


def boundary_loss(model, x_bc, y_bc, bc_regions=None):
    output = model(x_bc)

    u_pred = output[:, 0:1]
    v_pred = output[:, 1:2]

    u_real = y_bc[:, 0:1]
    v_real = y_bc[:, 1:2]

    if bc_regions is None:
        return (
            torch.mean((u_pred - u_real) ** 2)
            + torch.mean((v_pred - v_real) ** 2)
        )

    total = torch.tensor(0.0, device=x_bc.device)

    for region_name, (mask, weight) in bc_regions.items():
        if mask.sum() == 0:
            continue

        mask = mask.to(x_bc.device)

        err_u = torch.mean((u_pred[mask] - u_real[mask]) ** 2)
        err_v = torch.mean((v_pred[mask] - v_real[mask]) ** 2)

        total = total + weight * (err_u + err_v)

    return total


def boundary_loss_parts(model, x_bc, y_bc, bc_regions):
    """
    Solo diagnóstico. No se usa necesariamente para entrenar.
    """
    output = model(x_bc)

    u_pred = output[:, 0:1]
    v_pred = output[:, 1:2]

    u_real = y_bc[:, 0:1]
    v_real = y_bc[:, 1:2]

    parts = {}

    for region_name, (mask, weight) in bc_regions.items():
        if mask.sum() == 0:
            parts[f"bc_{region_name}"] = torch.tensor(float("nan"), device=x_bc.device)
            continue

        mask = mask.to(x_bc.device)

        err_u = torch.mean((u_pred[mask] - u_real[mask]) ** 2)
        err_v = torch.mean((v_pred[mask] - v_real[mask]) ** 2)

        parts[f"bc_{region_name}"] = err_u + err_v

    return parts


def total_loss(
    model,
    x_bc,
    y_bc,
    x_coll,
    y_coll,
    domain_cfg,
    loss_weights,
    bc_regions=None,
    normalized=False,
):
    physics_l = physics_loss(
        model,
        x_coll,
        y_coll,
        domain_cfg,
        normalized=normalized,
    )

    boundary_l = boundary_loss(
        model,
        x_bc,
        y_bc,
        bc_regions=bc_regions,
    )

    total = (
        loss_weights["residual"] * physics_l
        + loss_weights["bc"] * boundary_l
    )

    parts = {
        "physics_loss": physics_l,
        "boundary_loss": boundary_l,
        "w_residual": loss_weights["residual"],
        "w_bc": loss_weights["bc"],
    }

    if bc_regions is not None:
        parts.update(boundary_loss_parts(model, x_bc, y_bc, bc_regions))

    return total, parts

## 8. Dynamic loss weighting

Apagado por defecto para el baseline principal. Si se activa, usa norma global de todos los gradientes no nulos, no el primer gradiente.

In [ ]:
def _as_float(value):
    if torch.is_tensor(value):
        return float(value.detach().cpu().item())
    return float(value)


def grad_global_norm(loss, model):
    params = [p for p in model.parameters() if p.requires_grad]
    grads = torch.autograd.grad(loss, params, retain_graph=True, allow_unused=True)
    norms = [g.detach().norm(2) for g in grads if g is not None]
    if not norms:
        return torch.tensor(0.0, device=loss.device)
    return torch.norm(torch.stack(norms), p=2)


def update_loss_weights(model, physics_l, boundary_l, loss_weights, alpha=0.9, clamp_min=0.1, clamp_max=10.0):
    eps = 1e-12
    grad_physics = grad_global_norm(physics_l, model)
    grad_boundary = grad_global_norm(boundary_l, model)
    target = 0.5 * (grad_physics + grad_boundary)

    residual_weight = target / (grad_physics + eps)
    bc_weight = target / (grad_boundary + eps)

    new_residual = alpha * loss_weights['residual'] + (1.0 - alpha) * residual_weight
    new_bc = alpha * loss_weights['bc'] + (1.0 - alpha) * bc_weight

    return {
        'residual': torch.clamp(new_residual.detach(), clamp_min, clamp_max),
        'bc': torch.clamp(new_bc.detach(), clamp_min, clamp_max),
    }

## 9. Training utilities

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_optimizers(model, train_cfg):
    optimizer_adam = torch.optim.Adam(model.parameters(), lr=train_cfg.lr_adam)
    optimizer_lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=train_cfg.lr_lbfgs,
        max_iter=train_cfg.lbfgs_max_iter,
        history_size=train_cfg.lbfgs_history_size,
    )
    return optimizer_adam, optimizer_lbfgs


def get_experiment_dirs(experiment_id):
    model_dir = os.path.join(model_root, experiment_id)
    result_dir = os.path.join(results_root, experiment_id)
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(result_dir, exist_ok=True)
    return model_dir, result_dir


def serializable_model_config(config):
    return {k: v for k, v in config.items() if k != 'factory'}


def checkpoint_config_matches(ckpt, experiment_id, model_config):
    return (
        ckpt.get('experiment_id') == experiment_id
        and ckpt.get('model_config') == serializable_model_config(model_config)
    )


def fresh_loss_weights():
    return {'residual': torch.tensor(1.0, device=device), 'bc': torch.tensor(1.0, device=device)}


def save_checkpoint(path, model, optimizer_adam, optimizer_lbfgs, history, loss_weights,
                    experiment_id, model_config, phase, epoch):
    ckpt = {
        'experiment_id': experiment_id,
        'model_config': serializable_model_config(model_config),
        'phase': phase,
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_adam_state': optimizer_adam.state_dict(),
        'optimizer_lbfgs_state': optimizer_lbfgs.state_dict(),
        'history': history,
        'loss_weights': {k: _as_float(v) for k, v in loss_weights.items()},
    }
    torch.save(ckpt, path)


def load_checkpoint_if_exists(path, model, optimizer_adam, optimizer_lbfgs, experiment_id,
                              model_config, train_cfg):
    if not train_cfg.resume or not os.path.exists(path):
        return 0, 'adam', [], fresh_loss_weights()

    ckpt = torch.load(path, map_location=device)
    if not checkpoint_config_matches(ckpt, experiment_id, model_config):
        message = f'Checkpoint incompatible para {experiment_id}: {path}'
        if train_cfg.allow_incompatible_resume:
            print(message + '. Se empieza desde cero porque allow_incompatible_resume=True.')
            return 0, 'adam', [], fresh_loss_weights()
        raise RuntimeError(message)

    model.load_state_dict(ckpt['model_state'])
    optimizer_adam.load_state_dict(ckpt['optimizer_adam_state'])
    optimizer_lbfgs.load_state_dict(ckpt['optimizer_lbfgs_state'])
    history = ckpt.get('history', [])
    loss_weights = {
        'residual': torch.tensor(ckpt['loss_weights']['residual'], dtype=torch.float32, device=device),
        'bc': torch.tensor(ckpt['loss_weights']['bc'], dtype=torch.float32, device=device),
    }
    next_epoch = int(ckpt.get('epoch', -1)) + 1
    phase = ckpt.get('phase', 'adam')
    print(f'Resumiendo {experiment_id}: phase={phase}, next_epoch={next_epoch}')
    return next_epoch, phase, history, loss_weights


def history_row(experiment_id, phase, epoch, total, parts, elapsed_sec):
    row = {
        "experiment_id": experiment_id,
        "phase": phase,
        "epoch": int(epoch),
        "loss_total": _as_float(total),
        "physics_loss": _as_float(parts["physics_loss"]),
        "boundary_loss": _as_float(parts["boundary_loss"]),
        "w_residual": _as_float(parts["w_residual"]),
        "w_bc": _as_float(parts["w_bc"]),
        "elapsed_sec": float(elapsed_sec),
    }

    # Diagnóstico por región si existe
    for key in ["bc_inlet", "bc_top", "bc_bottom", "bc_circle"]:
        if key in parts:
            row[key] = _as_float(parts[key])

    return row


def evaluate_history(
    experiment_id,
    model_config,
    history,
    train_cfg,
    data_cfg,
    elapsed_sec,
    parameters,
    tensors,
):
    last = history[-1] if history else {}

    return {
        "experiment_id": experiment_id,
        "family": model_config["family"],
        "parameters": int(parameters),

        "final_total_loss": last.get("loss_total"),
        "final_physics_loss": last.get("physics_loss"),
        "final_boundary_loss": last.get("boundary_loss"),

        "final_bc_inlet": last.get("bc_inlet"),
        "final_bc_top": last.get("bc_top"),
        "final_bc_bottom": last.get("bc_bottom"),
        "final_bc_circle": last.get("bc_circle"),

        "final_w_residual": last.get("w_residual"),
        "final_w_bc": last.get("w_bc"),

        "epochs_completed": len(history),
        "training_time_min": elapsed_sec / 60.0,

        "normalized": model_config.get("normalized", False),
        "tensors": model_config.get("tensors", "phys"),

        "auto_weight_adam": train_cfg.auto_weight_adam,
        "auto_weight_lbfgs": train_cfg.auto_weight_lbfgs,

        "collocation_points": int(tensors["x_coll"].shape[0]),
        "boundary_points": int(tensors["x_bc"].shape[0]),

        "seed": data_cfg.seed,
    }


def save_artifacts(
    experiment_id,
    model,
    model_config,
    history,
    train_cfg,
    data_cfg,
    elapsed_sec,
    tensors,
):
    model_dir, result_dir = get_experiment_dirs(experiment_id)

    # Guardar historial completo como JSON para preservar columnas por región
    history_path = os.path.join(result_dir, "history.json")
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    # Guardar loss.npy compacto para plots rápidos.
    # Columnas:
    # 0 total, 1 physics, 2 boundary, 3 w_residual, 4 w_bc,
    # 5 bc_inlet, 6 bc_top, 7 bc_bottom, 8 bc_circle
    history_array = np.array([
        [
            row.get("loss_total", np.nan),
            row.get("physics_loss", np.nan),
            row.get("boundary_loss", np.nan),
            row.get("w_residual", np.nan),
            row.get("w_bc", np.nan),
            row.get("bc_inlet", np.nan),
            row.get("bc_top", np.nan),
            row.get("bc_bottom", np.nan),
            row.get("bc_circle", np.nan),
        ]
        for row in history
    ], dtype=np.float64)

    np.save(os.path.join(result_dir, "loss.npy"), history_array)

    # Guardar state_dict, no modelo completo
    torch.save(
        model.state_dict(),
        os.path.join(model_dir, "final_state_dict.pth"),
    )

    metrics = evaluate_history(
        experiment_id=experiment_id,
        model_config=model_config,
        history=history,
        train_cfg=train_cfg,
        data_cfg=data_cfg,
        elapsed_sec=elapsed_sec,
        parameters=count_parameters(model),
        tensors=tensors,
    )

    with open(os.path.join(result_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    config = {
        "experiment_id": experiment_id,
        "model_config": serializable_model_config(model_config),
        "domain_cfg": asdict(domain_cfg),
        "data_cfg": asdict(data_cfg),
        "train_cfg": asdict(train_cfg),
        "history_columns": [
            "loss_total",
            "physics_loss",
            "boundary_loss",
            "w_residual",
            "w_bc",
            "bc_inlet",
            "bc_top",
            "bc_bottom",
            "bc_circle",
        ],
    }

    with open(os.path.join(result_dir, "config.json"), "w") as f:
        json.dump(config, f, indent=2)

    return metrics


def evaluate_model(model, domain_cfg, n_test=300):
    model.eval()
    test_tensor, points = get_test_grid(domain_cfg, n_test=n_test)
    with torch.no_grad():
        pred = model(test_tensor).detach().cpu().numpy()
    return points, pred

In [ ]:
def train_one_experiment(experiment_id, config_map, domain_cfg, train_cfg, tensors):
    if experiment_id not in config_map:
        raise KeyError(
            f"Experimento no registrado: {experiment_id}. "
            f"Opciones: {list(config_map)}"
        )

    model_config = config_map[experiment_id]
    model = model_config["factory"]().to(device)

    optimizer_adam, optimizer_lbfgs = make_optimizers(model, train_cfg)

    model_dir, _ = get_experiment_dirs(experiment_id)
    checkpoint_path = os.path.join(model_dir, "checkpoint.pth")

    start_epoch, phase, history, loss_weights = load_checkpoint_if_exists(
        checkpoint_path,
        model,
        optimizer_adam,
        optimizer_lbfgs,
        experiment_id,
        model_config,
        train_cfg,
    )

    x_bc = tensors["x_bc"]
    y_bc = tensors["y_bc"]
    x_coll = tensors["x_coll"]
    y_coll = tensors["y_coll"]

    normalized = model_config.get("tensors") == "norm"

    bc_regions = build_bc_regions(
        x_bc,
        domain_cfg,
        data_cfg,
        normalized=normalized,
    )

    started = time.time()
    nan_aborted = False

    def compute_total_and_parts():
        return total_loss(
            model=model,
            x_bc=x_bc,
            y_bc=y_bc,
            x_coll=x_coll,
            y_coll=y_coll,
            domain_cfg=domain_cfg,
            loss_weights=loss_weights,
            bc_regions=bc_regions,
            normalized=normalized,
        )

    def loss_is_bad(loss):
        return not torch.isfinite(loss).all()

    def append_bad_row(phase_name, epoch_value, total_value, parts_value):
        row = history_row(
            experiment_id,
            phase_name,
            epoch_value,
            total_value,
            parts_value,
            time.time() - started,
        )
        row["status"] = "non_finite_loss"
        history.append(row)

    print("\n" + "=" * 80)
    print(f"Entrenando: {experiment_id}")
    print("=" * 80)
    print(f"  Familia      : {model_config.get('family')}")
    print(f"  Normalized   : {normalized}")
    print(f"  Phase start  : {phase}")
    print(f"  Epoch start  : {start_epoch}")
    print(f"  Params       : {count_parameters(model):,}")
    print(f"  Checkpoint   : {checkpoint_path}")

    # Si ya terminó, no reentrenar.
    if phase in ["done", "aborted"]:
        print(f"  Checkpoint en phase='{phase}'. No se reentrena.")
        elapsed_sec = time.time() - started

        metrics = save_artifacts(
            experiment_id,
            model,
            model_config,
            history,
            train_cfg,
            data_cfg,
            elapsed_sec,
            tensors,
        )
        return model, history, metrics

    # ------------------------------------------------------------------
    # ADAM
    # ------------------------------------------------------------------
    if phase == "adam":
        pbar = tqdm(
            range(start_epoch, train_cfg.epochs_adam),
            desc=f"Adam [{experiment_id}]",
            ncols=850,
        )

        for epoch in pbar:
            model.train()
            optimizer_adam.zero_grad()

            total, parts = compute_total_and_parts()

            if loss_is_bad(total):
                pbar.write(
                    f"  NaN/Inf @ época {epoch} [{experiment_id}] — Adam cortado."
                )
                append_bad_row("adam", epoch, total, parts)
                nan_aborted = True
                break

            # Dynamic loss weighting opcional, pero recalculando con mismos flags.
            if (
                train_cfg.auto_weight_adam
                and epoch > 0
                and epoch % train_cfg.loss_weight_update_frequency == 0
            ):
                loss_weights = update_loss_weights(
                    model,
                    parts["physics_loss"],
                    parts["boundary_loss"],
                    loss_weights,
                )

                total, parts = compute_total_and_parts()

                if loss_is_bad(total):
                    pbar.write(
                        f"  NaN/Inf después de auto-weight @ época {epoch} "
                        f"[{experiment_id}] — Adam cortado."
                    )
                    append_bad_row("adam", epoch, total, parts)
                    nan_aborted = True
                    break

            total.backward()
            optimizer_adam.step()

            row = history_row(
                experiment_id,
                "adam",
                epoch,
                total,
                parts,
                time.time() - started,
            )
            history.append(row)

            if epoch % 10 == 0:
                pbar.set_postfix({
                    "loss": f"{row['loss_total']:.3e}",
                    "phys": f"{row['physics_loss']:.3e}",
                    "bc": f"{row['boundary_loss']:.3e}",
                })

            if epoch > 0 and epoch % train_cfg.checkpoint_every == 0:
                save_checkpoint(
                    checkpoint_path,
                    model,
                    optimizer_adam,
                    optimizer_lbfgs,
                    history,
                    loss_weights,
                    experiment_id,
                    model_config,
                    "adam",
                    epoch,
                )

        if not nan_aborted:
            phase = "lbfgs"
            start_epoch = 0

            save_checkpoint(
                checkpoint_path,
                model,
                optimizer_adam,
                optimizer_lbfgs,
                history,
                loss_weights,
                experiment_id,
                model_config,
                "lbfgs",
                start_epoch,
            )

    # ------------------------------------------------------------------
    # L-BFGS
    # ------------------------------------------------------------------
    if phase == "lbfgs" and not nan_aborted:
        pbar = tqdm(
            range(start_epoch, train_cfg.epochs_lbfgs),
            desc=f"LBFGS [{experiment_id}]",
            ncols=850,
        )

        for epoch in pbar:
            model.train()

            def closure():
                optimizer_lbfgs.zero_grad()

                total_closure, parts_closure = compute_total_and_parts()

                if loss_is_bad(total_closure):
                    # L-BFGS no maneja bien excepciones dentro de closure.
                    # Devolvemos la loss tal cual; se detecta después del step.
                    return total_closure

                total_closure.backward()
                return total_closure

            _ = optimizer_lbfgs.step(closure)

            # Recalcular loss después del step. Esta es la métrica válida.
            optimizer_lbfgs.zero_grad()
            total_eval, parts_eval = compute_total_and_parts()

            if loss_is_bad(total_eval):
                pbar.write(
                    f"  NaN/Inf @ época {epoch} [{experiment_id}] — L-BFGS cortado."
                )
                append_bad_row("lbfgs", epoch, total_eval, parts_eval)
                nan_aborted = True
                break

            if (
                train_cfg.auto_weight_lbfgs
                and epoch > 0
                and epoch % train_cfg.loss_weight_update_frequency == 0
            ):
                loss_weights = update_loss_weights(
                    model,
                    parts_eval["physics_loss"],
                    parts_eval["boundary_loss"],
                    loss_weights,
                )

                total_eval, parts_eval = compute_total_and_parts()

                if loss_is_bad(total_eval):
                    pbar.write(
                        f"  NaN/Inf después de auto-weight @ época {epoch} "
                        f"[{experiment_id}] — L-BFGS cortado."
                    )
                    append_bad_row("lbfgs", epoch, total_eval, parts_eval)
                    nan_aborted = True
                    break

            row = history_row(
                experiment_id,
                "lbfgs",
                epoch,
                total_eval,
                parts_eval,
                time.time() - started,
            )
            history.append(row)

            if epoch % 10 == 0:
                pbar.set_postfix({
                    "loss": f"{row['loss_total']:.3e}",
                    "phys": f"{row['physics_loss']:.3e}",
                    "bc": f"{row['boundary_loss']:.3e}",
                })

            if epoch > 0 and epoch % train_cfg.checkpoint_every == 0:
                save_checkpoint(
                    checkpoint_path,
                    model,
                    optimizer_adam,
                    optimizer_lbfgs,
                    history,
                    loss_weights,
                    experiment_id,
                    model_config,
                    "lbfgs",
                    epoch,
                )

    # ------------------------------------------------------------------
    # CIERRE
    # ------------------------------------------------------------------
    elapsed_sec = time.time() - started
    final_phase = "aborted" if nan_aborted else "done"
    final_epoch = len(history) - 1

    save_checkpoint(
        checkpoint_path,
        model,
        optimizer_adam,
        optimizer_lbfgs,
        history,
        loss_weights,
        experiment_id,
        model_config,
        final_phase,
        final_epoch,
    )

    if nan_aborted:
        print(f"  [{experiment_id}] Abortado por NaN/Inf tras {final_epoch} pasos.")
    else:
        print(f"  [{experiment_id}] Entrenamiento completado.")

    metrics = save_artifacts(
        experiment_id,
        model,
        model_config,
        history,
        train_cfg,
        data_cfg,
        elapsed_sec,
        tensors,
    )

    return model, history, metrics

## 10. Sanity checks antes de entrenar

In [ ]:
def run_sanity_checks(experiment_id, config_map, domain_cfg, tensors):
    if experiment_id not in config_map:
        raise KeyError(f"Experimento no registrado: {experiment_id}")

    model_config = config_map[experiment_id]
    model = model_config["factory"]().to(device)

    normalized = model_config.get("tensors") == "norm"

    x_bc = tensors["x_bc"]
    y_bc = tensors["y_bc"]
    x_coll = tensors["x_coll"]
    y_coll = tensors["y_coll"]

    bc_regions = build_bc_regions(
        x_bc,
        domain_cfg,
        data_cfg,
        normalized=normalized,
    )

    params = count_parameters(model)

    with torch.no_grad():
        dummy = torch.randn(4, 2, device=device)
        out = model(dummy)

    loss_weights = fresh_loss_weights()

    total, parts = total_loss(
        model=model,
        x_bc=x_bc,
        y_bc=y_bc,
        x_coll=x_coll,
        y_coll=y_coll,
        domain_cfg=domain_cfg,
        loss_weights=loss_weights,
        bc_regions=bc_regions,
        normalized=normalized,
    )

    print(f"  Device              : {device}")
    print(f"  Boundary points     : {x_bc.shape[0]:,}")
    print(f"  Collocation points  : {x_coll.shape[0]:,}")
    print(f"  Model parameters    : {params:,}")
    print(f"  Forward dummy       : {tuple(dummy.shape)} -> {tuple(out.shape)}")
    print(f"  Normalized tensors  : {normalized}")

    print(f"  Initial total loss  : {_as_float(total):.6e}")
    print(f"  Initial physics loss: {_as_float(parts['physics_loss']):.6e}")
    print(f"  Initial BC loss     : {_as_float(parts['boundary_loss']):.6e}")

    for key in ["bc_inlet", "bc_top", "bc_bottom", "bc_circle"]:
        if key in parts:
            print(f"  Initial {key:<10}: {_as_float(parts[key]):.6e}")

    if out.shape != (4, 3):
        raise RuntimeError(f"Forward output incorrecto: esperado (4,3), obtenido {tuple(out.shape)}")

    if not torch.isfinite(total):
        raise FloatingPointError("Initial total loss no es finita.")

    if not torch.isfinite(parts["physics_loss"]):
        raise FloatingPointError("Initial physics loss no es finita.")

    if not torch.isfinite(parts["boundary_loss"]):
        raise FloatingPointError("Initial boundary loss no es finita.")

    print("  ✓ Sanity check OK")

## 11. Run experiment

Ruta principal reproducible. No hay dropdown. Usa `EXPERIMENT_ID` para una corrida individual o `RUN_QUEUE` para correr el batch completo sin intervencion manual.

In [ ]:
import os
import shutil

RESET_EXPERIMENTS = True

if RESET_EXPERIMENTS:
    for exp_id in MODEL_CONFIGS.keys():
        model_dir = os.path.join(model_root, exp_id)
        result_dir = os.path.join(results_root, exp_id)

        if os.path.exists(model_dir):
            shutil.rmtree(model_dir)
            print(f"Eliminado model_dir: {model_dir}")

        if os.path.exists(result_dir):
            shutil.rmtree(result_dir)
            print(f"Eliminado result_dir: {result_dir}")

    print("Experimentos reiniciados desde cero.")

In [ ]:
EXPERIMENT_ID = 'FCN'
RUN_QUEUE     = ['FCN','PIKAN-d5','cPIKAN-d5']
RUN_BATCH     = True

tensors_map = {'phys': tensors_phys, 'norm': tensors_norm}

experiments_to_run = RUN_QUEUE if RUN_BATCH else [EXPERIMENT_ID]
run_results   = []
trained_models = {}

for experiment_id in experiments_to_run:
    cfg      = MODEL_CONFIGS[experiment_id]
    tensors  = tensors_map[cfg.get('tensors', 'phys')]

    model, history, metrics = train_one_experiment(
        experiment_id, MODEL_CONFIGS, domain_cfg, train_cfg, tensors,
    )
    trained_models[experiment_id] = model
    run_results.append(metrics)

run_results

## 12. Evaluation

Tabla final basada en artefactos y metricas de cada experimento.

In [ ]:
import os
import json
import math
import uuid
import pandas as pd
from IPython.display import display, HTML


def load_metrics_table(results_root, experiment_ids):
    rows = []

    for experiment_id in experiment_ids:
        metrics_path = os.path.join(results_root, experiment_id, "metrics.json")

        if not os.path.exists(metrics_path):
            print(f"Sin metrics.json: {experiment_id}")
            continue

        with open(metrics_path, "r") as f:
            row = json.load(f)

        rows.append(row)

    return rows


def display_metrics_table(rows):
    if not rows:
        print("No hay métricas guardadas todavía.")
        return

    display_cols = {
        "experiment_id": "Experimento",
        "family": "Familia",
        "parameters": "Params",
        "normalized": "Norm.",
        "tensors": "Tensors",
        "collocation_points": "N coll",

        "final_total_loss": "Loss total",
        "final_physics_loss": "Loss física",
        "final_boundary_loss": "Loss BC",

        "final_bc_inlet": "BC inlet",
        "final_bc_top": "BC top",
        "final_bc_bottom": "BC bottom",
        "final_bc_circle": "BC cilindro",

        "epochs_completed": "Épocas",
        "training_time_min": "Tiempo min",
    }

    df = pd.DataFrame(rows)

    for col in display_cols:
        if col not in df.columns:
            df[col] = None

    df = df[list(display_cols.keys())].rename(columns=display_cols)

    def is_missing(val):
        return val is None or (isinstance(val, float) and math.isnan(val))

    def fmt_loss(val):
        if is_missing(val):
            return "—"

        try:
            f = float(val)

            if math.isnan(f):
                return '<span class="metric-bad">NaN</span>'

            if math.isinf(f):
                return '<span class="metric-bad">Inf</span>'

            return f"{f:.3e}"
        except Exception:
            return str(val)

    def fmt_int(val):
        if is_missing(val):
            return "—"
        try:
            return f"{int(val):,}"
        except Exception:
            return "—"

    def fmt_float_1(val):
        if is_missing(val):
            return "—"
        try:
            return f"{float(val):.1f}"
        except Exception:
            return "—"

    loss_cols = [
        "Loss total",
        "Loss física",
        "Loss BC",
        "BC inlet",
        "BC top",
        "BC bottom",
        "BC cilindro",
    ]

    for col in loss_cols:
        df[col] = df[col].apply(fmt_loss)

    df["Params"] = df["Params"].apply(fmt_int)
    df["N coll"] = df["N coll"].apply(fmt_int)
    df["Épocas"] = df["Épocas"].apply(fmt_int)
    df["Tiempo min"] = df["Tiempo min"].apply(fmt_float_1)

    # ID único para evitar que el CSS afecte otras tablas/output del notebook
    table_id = f"metrics_table_{uuid.uuid4().hex[:8]}"

    table_html = df.to_html(
        escape=False,
        index=False,
        border=0,
        classes="metrics-table-inner",
    )

    html = f"""
    <style>
      #{table_id} {{
        width: 100%;
        max-width: 100%;
        overflow-x: auto;
        overflow-y: hidden;
        margin: 12px 0 18px 0;
        border: 1px solid #d8dee9;
        border-radius: 10px;
        background: #ffffff;
      }}

      #{table_id} table.metrics-table-inner {{
        border-collapse: collapse;
        width: max-content;
        min-width: 100%;
        table-layout: auto;
        font-family: Arial, sans-serif;
        font-size: 12px;
        line-height: 1.25;
      }}

      #{table_id} table.metrics-table-inner thead tr {{
        background: #111827;
        color: #ffffff;
      }}

      #{table_id} table.metrics-table-inner thead th {{
        padding: 8px 10px;
        text-align: left;
        font-size: 10px;
        font-weight: 700;
        letter-spacing: 0.04em;
        text-transform: uppercase;
        white-space: nowrap;
        border-bottom: 1px solid #374151;
      }}

      #{table_id} table.metrics-table-inner tbody tr {{
        border-bottom: 1px solid #e5e7eb;
      }}

      #{table_id} table.metrics-table-inner tbody tr:nth-child(even) {{
        background: #f9fafb;
      }}

      #{table_id} table.metrics-table-inner tbody tr:hover {{
        background: #eef2ff;
      }}

      #{table_id} table.metrics-table-inner tbody td {{
        padding: 7px 10px;
        color: #111827;
        vertical-align: middle;
        white-space: nowrap;
        text-align: left;
      }}

      #{table_id} table.metrics-table-inner tbody td:first-child {{
        font-weight: 700;
      }}

      #{table_id} .metric-bad {{
        color: #b91c1c;
        font-weight: 700;
      }}
    </style>

    <div id="{table_id}">
      {table_html}
    </div>
    """

    display(HTML(html))


metrics_rows = load_metrics_table(results_root, MODEL_CONFIGS.keys())
display_metrics_table(metrics_rows)

## 13. Plots

El plotting se mantiene separado del entrenamiento. Las comparaciones leen artefactos guardados, no variables transitorias.

###  Curvas de pérdida — todos los experimentos

In [ ]:
def plot_loss_comparison(results_root, model_configs):
    COLORS = {
        'FCN'        : '#1d4ed8',
        'PIKAN-NORM' : '#dc2626',
        'cPIKAN-NORM': '#16a34a',
    }
    DEFAULT_COLORS = ['#7c3aed', '#d97706', '#0891b2']

    fig, axes = plt.subplots(1, 3, figsize=(20, 4))
    fig.patch.set_facecolor('white')

    loss_keys  = ['loss_total', 'physics_loss', 'boundary_loss']
    loss_titles = ['Loss total', 'Loss física (residual NS)', 'Loss BC']

    all_series = {}
    for exp_id in model_configs:
        loss_path = os.path.join(results_root, exp_id, 'loss.npy')
        if not os.path.exists(loss_path):
            continue
        arr = np.load(loss_path)
        if arr.size == 0:
            continue
        all_series[exp_id] = arr

    if not all_series:
        print('No hay curvas guardadas. Entrenar al menos un experimento primero.')
        return

    color_list = list(COLORS.values()) + DEFAULT_COLORS
    exp_colors = {
        exp_id: COLORS.get(exp_id, color_list[i % len(color_list)])
        for i, exp_id in enumerate(all_series)
    }

    for ax, col_idx, title in zip(axes, range(3), loss_titles):
        ax.set_facecolor('white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#cbd5e1')
        ax.spines['bottom'].set_color('#cbd5e1')
        ax.tick_params(colors='#475569', labelsize=9)
        ax.set_xlabel('Step', color='#475569', fontsize=9)
        ax.set_ylabel('Loss', color='#475569', fontsize=9)
        ax.set_title(title, fontsize=11, fontweight='bold', color='#0f172a', pad=8)
        ax.grid(True, alpha=0.25, color='#94a3b8', linewidth=0.7)

        for exp_id, arr in all_series.items():
            if arr.shape[1] <= col_idx:
                continue
            series = arr[:, col_idx]
            ax.plot(series, lw=1.8, color=exp_colors[exp_id],
                    label=exp_id, alpha=0.9)

        ax.set_yscale('log')
        ax.legend(fontsize=9, framealpha=0.9, edgecolor='#e2e8f0',
                  facecolor='white', labelcolor='#0f172a')

    fig.suptitle('Comparación de curvas de pérdida — NS cilindro 2D',
                 fontsize=13, fontweight='bold', color='#0f172a', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(results_root, 'compare_loss.jpeg'),
                format='pdf', bbox_inches='tight', facecolor='white')
    plt.show()

plot_loss_comparison(results_root, MODEL_CONFIGS)

### Curva de pérdida — experimento activo


In [ ]:
def plot_loss_curves_individual(results_root, model_configs):
    """
    Un subplot por experimento, lado a lado.
    Lee history desde loss.npy — no depende de variables en memoria.
    Cada subplot muestra total, física y BC con marcador Adam→LBFGS.
    """
    available = {}
    for exp_id in model_configs:
        loss_path = os.path.join(results_root, exp_id, 'loss.npy')
        if os.path.exists(loss_path):
            arr = np.load(loss_path, allow_pickle=True)
            if arr.size > 0:
                available[exp_id] = arr

    if not available:
        print('No hay curvas guardadas. Entrenar al menos un experimento.')
        return

    n = len(available)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 4), sharey=False)
    fig.patch.set_facecolor('white')
    if n == 1:
        axes = [axes]

    COLORS = {
        'total'   : '#1d4ed8',
        'physics' : '#dc2626',
        'bc'      : '#16a34a',
        'lbfgs'   : '#94a3b8',
    }

    for ax, (exp_id, arr) in zip(axes, available.items()):
        ax.set_facecolor('white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#cbd5e1')
        ax.spines['bottom'].set_color('#cbd5e1')

        # loss.npy columnas: [total, physics, boundary]
        # Si fue guardado como array de dicts, extraer columnas
        if arr.dtype == object:
            history = arr.tolist()
            total    = [r['loss_total']    for r in history]
            physics  = [r['physics_loss']  for r in history]
            boundary = [r['boundary_loss'] for r in history]
            phases   = [r.get('phase', '') for r in history]
            lbfgs_start = next(
                (i for i, p in enumerate(phases) if p == 'lbfgs'), None
            )
        else:
            total    = arr[:, 0].tolist()
            physics  = arr[:, 1].tolist() if arr.shape[1] > 1 else None
            boundary = arr[:, 2].tolist() if arr.shape[1] > 2 else None
            lbfgs_start = None

            # Inferir corte Adam/LBFGS desde metrics.json si existe
            metrics_path = os.path.join(results_root, exp_id, 'metrics.json')
            if os.path.exists(metrics_path):
                with open(metrics_path) as f:
                    m = json.load(f)
                epochs_adam = m.get('epochs_adam', None)
                if epochs_adam:
                    lbfgs_start = int(epochs_adam)

        steps = list(range(len(total)))

        ax.plot(steps, total, lw=2, color=COLORS['total'],
                label='Total', alpha=0.95)
        if physics:
            ax.plot(steps, physics, lw=1.5, color=COLORS['physics'],
                    label='Física', alpha=0.85)
        if boundary:
            ax.plot(steps, boundary, lw=1.5, color=COLORS['bc'],
                    label='BC', alpha=0.85)

        if lbfgs_start and lbfgs_start < len(steps):
            ax.axvline(x=lbfgs_start, color=COLORS['lbfgs'],
                       lw=1.2, linestyle='--', alpha=0.8)
            ax.text(lbfgs_start + max(5, len(steps) * 0.01),
                    ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 1e-10,
                    'L-BFGS', fontsize=7, color='#64748b',
                    va='bottom')

        ax.set_yscale('log')
        ax.set_title(exp_id, fontsize=11, fontweight='bold',
                     color='#0f172a', pad=7)
        ax.set_xlabel('Step', color='#475569', fontsize=9)
        ax.set_ylabel('Loss', color='#475569', fontsize=9)
        ax.tick_params(colors='#475569', labelsize=8)
        ax.grid(True, alpha=0.2, color='#94a3b8', linewidth=0.6)
        ax.legend(fontsize=8, framealpha=0.9, edgecolor='#e2e8f0',
                  facecolor='white', labelcolor='#0f172a')

        # Anotar loss final
        final = total[-1]
        ax.annotate(
            f'final: {final:.2e}',
            xy=(len(steps) - 1, final),
            xytext=(-60, 12),
            textcoords='offset points',
            fontsize=8, color='#1d4ed8',
            arrowprops=dict(arrowstyle='->', color='#94a3b8', lw=0.8),
        )

    fig.suptitle('Curvas de pérdida individuales — NS cilindro 2D',
                 fontsize=13, fontweight='bold', color='#0f172a', y=1.03)
    plt.tight_layout()
    plt.savefig(os.path.join(results_root, 'loss_individual.jpeg'),
                format='pdf', bbox_inches='tight', facecolor='white')
    plt.show()

plot_loss_curves_individual(results_root, MODEL_CONFIGS)

### Campos Predichos

In [ ]:
from scipy.interpolate import griddata

def get_test_grid(domain_cfg, nx=220, ny=80):
    x_lin = np.linspace(0, domain_cfg.width,  nx)
    y_lin = np.linspace(0, domain_cfg.height, ny)
    Xi, Yi = np.meshgrid(x_lin, y_lin)
    x_flat = Xi.flatten()[:, None]
    y_flat = Yi.flatten()[:, None]
    mask   = ~inside_circle_condition(
        domain_cfg.circle_center_x, domain_cfg.circle_center_y,
        domain_cfg.circle_radius, x_flat[:, 0], y_flat[:, 0],
    )
    pts    = np.hstack((x_flat[mask], y_flat[mask]))
    tensor = torch.tensor(pts, dtype=torch.float32, device=device)
    return tensor, pts, (Xi, Yi)


def plot_all_fields(trained_models, model_configs, domain_cfg,
                    results_root, checkpoints_root=None):

    # ── Cargar desde checkpoint si no están en memoria ────────────
    to_plot = dict(trained_models)
    if checkpoints_root:
        for exp_id, cfg in model_configs.items():
            if exp_id in to_plot:
                continue
            ckpt_path = os.path.join(checkpoints_root, exp_id, 'checkpoint.pth')
            if not os.path.exists(ckpt_path):
                continue
            try:
                m = cfg['factory']().to(device)
                ckpt = torch.load(ckpt_path, map_location=device)
                m.load_state_dict(ckpt['model_state'])
                m.eval()
                to_plot[exp_id] = m
                print(f'  Cargado: {exp_id}')
            except Exception as e:
                print(f'  No se pudo cargar {exp_id}: {e}')

    if not to_plot:
        print('No hay modelos disponibles.')
        return

    # ── Malla compartida ──────────────────────────────────────────
    tensor_phys, pts_phys, (Xi, Yi) = get_test_grid(domain_cfg, nx=220, ny=80)
    cyl_mask = inside_circle_condition(
        domain_cfg.circle_center_x, domain_cfg.circle_center_y,
        domain_cfg.circle_radius, Xi, Yi,
    )
    theta = np.linspace(0, 2 * np.pi, 300)
    xc = domain_cfg.circle_center_x + domain_cfg.circle_radius * np.cos(theta)
    yc = domain_cfg.circle_center_y + domain_cfg.circle_radius * np.sin(theta)

    field_defs = [
        (0, r'$u^{(x)}$ — Velocidad horizontal',  'turbo'),
        (1, r'$u^{(y)}$ — Velocidad vertical',    'seismic'),
        (2, r'$P - \bar{P}$ — Presión relativa',  'RdBu_r'),
    ]
    fnames = ['ux', 'uy', 'p']

    # ── Paso 1: recolectar predicciones e interpolar ──────────────
    all_preds = {}
    all_grids = {}

    for exp_id, model in to_plot.items():
        cfg        = model_configs.get(exp_id, {})
        normalized = cfg.get('tensors') == 'norm'

        model.eval()
        with torch.no_grad():
            if normalized:
                pts_in       = pts_phys.copy()
                pts_in[:, 0] = 2.0 * pts_in[:, 0] / domain_cfg.width  - 1.0
                pts_in[:, 1] = 2.0 * pts_in[:, 1] / domain_cfg.height - 1.0
                tensor_in    = torch.tensor(
                    pts_in, dtype=torch.float32, device=device
                )
            else:
                tensor_in = tensor_phys
            pred = model(tensor_in).cpu().numpy()

        pred = np.nan_to_num(pred, nan=0.0)
        all_preds[exp_id] = pred

        for idx, _, _ in field_defs:
            Zi = griddata(pts_phys, pred[:, idx], (Xi, Yi), method='linear')
            Zi[cyl_mask] = np.nan
            # Opción A: centrar presión en media del dominio
            if idx == 2:
                Zi = Zi - np.nanmean(Zi)
            all_grids[(exp_id, idx)] = Zi

    # ── Paso 2: rangos globales — presión ya centrada ─────────────
    global_range = {}
    for idx, _, _ in field_defs:
        all_vals = np.concatenate([
            all_grids[(exp_id, idx)][~np.isnan(all_grids[(exp_id, idx)])]
            for exp_id in to_plot
        ])
        global_range[idx] = (
            np.percentile(all_vals, 2),
            np.percentile(all_vals, 98),
        )

    print('  Rangos globales por campo:')
    labels = {0: 'u(x)', 1: 'u(y)', 2: 'P-P̄'}
    for idx, (vmin, vmax) in global_range.items():
        print(f'    {labels[idx]:>5}: [{vmin:.4f}, {vmax:.4f}]')

    # ── Paso 3: graficar con escala compartida ────────────────────
    for exp_id in to_plot:
        fig, axes = plt.subplots(3, 1, figsize=(22, 9))
        fig.patch.set_facecolor('white')
        fig.suptitle(f'Campos predichos — {exp_id}',
                     fontsize=13, fontweight='bold', color='#0f172a', y=1.01)

        for ax, (idx, title, cmap), fname in zip(axes, field_defs, fnames):
            Zi         = all_grids[(exp_id, idx)]
            vmin, vmax = global_range[idx]

            # Para presión divergente: forzar simetría alrededor de cero
            if idx == 2:
                abs_max = max(abs(vmin), abs(vmax))
                vmin, vmax = -abs_max, abs_max

            ax.set_facecolor('#f8fafc')
            im = ax.imshow(
                Zi,
                origin='lower',
                cmap=cmap,
                extent=[0, domain_cfg.width, 0, domain_cfg.height],
                aspect='auto',
                vmin=vmin, vmax=vmax,
                interpolation='bilinear',
            )
            cb = plt.colorbar(im, ax=ax, fraction=0.012, pad=0.01)
            cb.ax.tick_params(labelsize=8, colors='#475569')
            cb.set_label(title.split('—')[0].strip(),
                         color='#0f172a', fontsize=9)

            ax.fill(xc, yc, color='#1e293b', alpha=0.95, zorder=5)
            ax.plot(xc, yc, color='white',   lw=0.8,     zorder=6)

            ax.set_title(title, fontsize=10, color='#0f172a',
                         fontweight='bold', pad=5)
            ax.set_xlabel('X', color='#475569', fontsize=9)
            ax.set_ylabel('Y', color='#475569', fontsize=9)
            ax.tick_params(colors='#475569', labelsize=8)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#e2e8f0')
            ax.spines['bottom'].set_color('#e2e8f0')
            ax.set_xlim(0, domain_cfg.width)
            ax.set_ylim(0, domain_cfg.height)

        plt.tight_layout()
        out = os.path.join(results_root, f'fields_{exp_id}.png')
        plt.savefig(out, bbox_inches='tight', facecolor='white', dpi=150)
        print(f'  Guardado: {out}')
        plt.show()


plot_all_fields(
    trained_models,
    MODEL_CONFIGS,
    domain_cfg,
    results_root=results_root,
    checkpoints_root=model_root,
)

### Diagnosticos

In [ ]:
def diagnose_bc_error(model, x_bc, y_bc, domain_cfg, data_cfg, normalized=False):
    model.eval()

    bc_regions = build_bc_regions(
        x_bc,
        domain_cfg,
        data_cfg,
        normalized=normalized,
    )

    with torch.no_grad():
        output = model(x_bc)

    u_pred = output[:, 0:1]
    v_pred = output[:, 1:2]

    u_real = y_bc[:, 0:1]
    v_real = y_bc[:, 1:2]

    print(f'\n{"Región":<12} {"N pts":>6} {"MSE u":>12} {"MSE v":>12}')
    print("-" * 48)

    for name, (mask, weight) in bc_regions.items():
        if mask.sum() == 0:
            print(f"{name:<12} {0:>6} {'—':>12} {'—':>12}")
            continue

        mask = mask.to(x_bc.device)

        mse_u = torch.mean((u_pred[mask] - u_real[mask]) ** 2).item()
        mse_v = torch.mean((v_pred[mask] - v_real[mask]) ** 2).item()

        print(f"{name:<12} {int(mask.sum()):>6} {mse_u:>12.4e} {mse_v:>12.4e}")


def pressure_drop(model, domain_cfg, normalized=False):
    cx = domain_cfg.circle_center_x
    cy = domain_cfg.circle_center_y
    r = domain_cfg.circle_radius

    def make_point(x_val, y_val):
        pt = np.array([[x_val, y_val]], dtype=np.float32)

        if normalized:
            pt[:, 0] = 2.0 * pt[:, 0] / domain_cfg.width - 1.0
            pt[:, 1] = 2.0 * pt[:, 1] / domain_cfg.height - 1.0

        return torch.tensor(pt, dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        p_front = model(make_point(cx - 1.1 * r, cy))[0, 2].item()
        p_back = model(make_point(cx + 1.1 * r, cy))[0, 2].item()

    delta_p = p_front - p_back

    print(f"\np frente cilindro : {p_front:.6e}")
    print(f"p detrás cilindro : {p_back:.6e}")
    print(f"ΔP frente-detrás  : {delta_p:.6e}")

    if delta_p > 0:
        print("Signo de ΔP: correcto")
    else:
        print("Signo de ΔP: invertido o no aprendido")


for exp_id, model in trained_models.items():
    cfg = MODEL_CONFIGS[exp_id]
    normalized = cfg.get("tensors") == "norm"
    tensors = tensors_norm if normalized else tensors_phys

    print("\n" + "=" * 70)
    print(exp_id)
    print("=" * 70)

    diagnose_bc_error(
        model,
        tensors["x_bc"],
        tensors["y_bc"],
        domain_cfg,
        data_cfg,
        normalized=normalized,
    )

    pressure_drop(
        model,
        domain_cfg,
        normalized=normalized,
    )

In [ ]:
import os
import shutil
import json
from datetime import datetime

# EXPORT CONFIG

PROJECT_DIR = "/kaggle/working/GradProject"

MODELS_DIR = os.path.join(PROJECT_DIR, "NS_models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "NS_results")

EXPORT_NAME = "NS_PIKANS_KAGGLE_EXPORT"
EXPORT_DIR = f"/kaggle/working/{EXPORT_NAME}"
ZIP_BASE = f"/kaggle/working/{EXPORT_NAME}"

# VALIDATION

assert os.path.exists(PROJECT_DIR), f"No existe PROJECT_DIR: {PROJECT_DIR}"
assert os.path.exists(MODELS_DIR), f"No existe MODELS_DIR: {MODELS_DIR}"
assert os.path.exists(RESULTS_DIR), f"No existe RESULTS_DIR: {RESULTS_DIR}"

# CREATE CLEAN EXPORT FOLDER


if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

os.makedirs(EXPORT_DIR, exist_ok=True)

# Copiar modelos y resultados
shutil.copytree(MODELS_DIR, os.path.join(EXPORT_DIR, "NS_models"))
shutil.copytree(RESULTS_DIR, os.path.join(EXPORT_DIR, "NS_results"))

# Metadata del export
export_metadata = {
    "export_name": EXPORT_NAME,
    "created_at": datetime.now().isoformat(),
    "source_project_dir": PROJECT_DIR,
    "models_dir": MODELS_DIR,
    "results_dir": RESULTS_DIR,
    "experiments_models": sorted(os.listdir(MODELS_DIR)),
    "experiments_results": sorted(os.listdir(RESULTS_DIR)),
}

with open(os.path.join(EXPORT_DIR, "export_metadata.json"), "w") as f:
    json.dump(export_metadata, f, indent=2)

# ZIP EXPORT

zip_file = shutil.make_archive(
    base_name=ZIP_BASE,
    format="zip",
    root_dir="/kaggle/working",
    base_dir=EXPORT_NAME,
)

print("Export creado correctamente.")
print("ZIP:", zip_file)

print("\nContenido exportado:")
for root, dirs, files in os.walk(EXPORT_DIR):
    level = root.replace(EXPORT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:10]:
        print(f"{indent}  {file}")